# Personality Knowledge Graph - Custom Analysis**Customizable notebook for analyzing different novels**---## 📚 Available NovelsConfigure the `NOVEL_SELECTION` below to analyze:- `dune` - Frank Herbert's Dune (sci-fi, politics, religion)- `bladerunner` - Philip K. Dick's Do Androids Dream of Electric Sheep (dystopia, identity)- `foundation` - Isaac Asimov's Foundation (space opera, psychohistory)- `neuromancer` - William Gibson's Neuromancer (cyberpunk, AI)- `dune2` - Dune Messiah (sequel)**Note**: You must run the pipeline first to generate outputs.

In [ ]:
# ============================================# CONFIGURATION: Change this to analyze different novels# ============================================NOVEL_SELECTION = "dune"  # Options: dune, bladerunner, foundation, neuromancer, dune2# Optional: Specify exact output directory (leave None to auto-detect latest)CUSTOM_OUTPUT_DIR = None  # e.g., "outputs/foundation_run_20251020_120000"# ============================================

---## 🚀 Quick Start Guide### Step 1: Run Pipeline (if not done yet)```bash# In terminal:cd "/Users/chromatrical/CAREER/Side Projects/Intellumia shortlist/Project"source .venv/bin/activateexport ANTHROPIC_API_KEY="your-key"# Process your chosen novel (50 passages ~5 min, ~$0.50)./scripts/process_individual.sh bladerunner 50# or./scripts/process_individual.sh foundation 50```### Step 2: Run This NotebookExecute all cells to analyze the results!

In [ ]:
# Install dependenciesimport subprocessimport sysdef install_if_missing(package):    try:        __import__(package)    except ImportError:        print(f"Installing {package}...")        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])packages = ['python-dotenv', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx']for pkg in packages:    install_if_missing(pkg.replace('-', '_') if '-' in pkg else pkg)print("✓ All dependencies installed")

In [ ]:
# Setup pathsfrom pathlib import Pathimport sysimport globproject_root = Path("/Users/chromatrical/CAREER/Side Projects/Intellumia shortlist/Project")sys.path.insert(0, str(project_root / "src"))# Novel metadataNOVEL_INFO = {    "dune": {        "title": "Dune",        "author": "Frank Herbert",        "genre": "Science Fiction",        "description": "Political intrigue on desert planet Arrakis"    },    "bladerunner": {        "title": "Do Androids Dream of Electric Sheep?",        "author": "Philip K. Dick",        "genre": "Dystopian Sci-Fi",        "description": "Bounty hunter tracks rogue androids in post-apocalyptic Earth"    },    "foundation": {        "title": "Foundation",        "author": "Isaac Asimov",        "genre": "Space Opera",        "description": "Mathematician predicts fall of Galactic Empire"    },    "neuromancer": {        "title": "Neuromancer",        "author": "William Gibson",        "genre": "Cyberpunk",        "description": "Hacker hired for impossible cyberspace heist"    },    "dune2": {        "title": "Dune Messiah",        "author": "Frank Herbert",        "genre": "Science Fiction",        "description": "Paul Atreides rules as Emperor, faces conspiracy"    }}# Auto-detect output directoryif CUSTOM_OUTPUT_DIR:    OUTPUT_DIR = project_root / CUSTOM_OUTPUT_DIRelse:    pattern = str(project_root / "outputs" / f"{NOVEL_SELECTION}_run_*")    matches = sorted(glob.glob(pattern), reverse=True)    if not matches:        raise FileNotFoundError(            f"No outputs found for '{NOVEL_SELECTION}'.\n"            f"Run: ./scripts/process_individual.sh {NOVEL_SELECTION} 50"        )    OUTPUT_DIR = Path(matches[0])novel_info = NOVEL_INFO[NOVEL_SELECTION]print(f"✓ Novel: {novel_info['title']} by {novel_info['author']}")print(f"✓ Genre: {novel_info['genre']}")print(f"✓ Output: {OUTPUT_DIR}")

In [ ]:
# Import librariesimport jsonimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport networkx as nxfrom collections import Counterfrom IPython.display import display, HTML, Markdownimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")print("✓ Libraries loaded")

In [ ]:
# Load pipeline outputsfrom pipeline.io_utils import load_jsonl, load_graphmltriples = load_jsonl(OUTPUT_DIR / "triples_canonical.jsonl")traits = load_jsonl(OUTPUT_DIR / "traits_final.jsonl")G = load_graphml(OUTPUT_DIR / "graph.graphml")with open(OUTPUT_DIR / "metrics.json") as f:    metrics = json.load(f)print(f"✓ Loaded {len(triples)} triples")print(f"✓ Loaded {len(traits)} personality profiles")print(f"✓ Loaded graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

---## 📊 Analysis Results

In [ ]:
# Novel overviewdisplay(Markdown(f"""# {novel_info['title']}**Author**: {novel_info['author']}  **Genre**: {novel_info['genre']}  **Description**: {novel_info['description']}---### Extraction Summary| Metric | Value ||--------|-------|| **Entities (nodes)** | {G.number_of_nodes():,} || **Relationships (edges)** | {G.number_of_edges():,} || **Unique relations** | {len(set(t['relation'] for t in triples))} || **Character profiles** | {len(traits)} || **Avg confidence** | {np.mean([t['confidence'] for t in triples]):.3f} || **Evidence coverage** | {metrics['evidence_quality']['evidence_coverage']:.1%} |"""))

### Sample Extracted Triples

In [ ]:
# Display sample triplesdf_triples = pd.DataFrame([{    "Subject": t["subject"],    "Relation": t["relation"],    "Object": t["object"],    "Confidence": f"{t['confidence']:.2f}",    "Evidence": t["evidence_span"]["text"][:60] + "..."} for t in triples[:10]])display(df_triples)

### Character Personality Profiles

In [ ]:
# Display all profilesdf_profiles = pd.DataFrame([{    "Character": p["person_name"],    "Traits": len(p["traits"]),    "Avg Confidence": f"{np.mean([t['confidence'] for t in p['traits']]):.2f}",    "Trait Names": ", ".join([t["trait_name"] for t in p["traits"]])} for p in traits])display(df_profiles)

### Detailed Profile: Main Character

In [ ]:
# Find character with most traitsmain_char = max(traits, key=lambda p: len(p["traits"]))display(Markdown(f"## {main_char['person_name']}"))df_main = pd.DataFrame([{    "Trait": t["trait_name"].capitalize(),    "Score": f"{t['score']:.2f}",    "Confidence": f"{t['confidence']:.2f}",    "Evidence Count": len(t.get("evidence_spans", []))} for t in main_char["traits"]])display(df_main)# Plot profileif main_char["traits"]:    fig, ax = plt.subplots(figsize=(10, 6))    trait_names = [t["trait_name"].capitalize() for t in main_char["traits"]]    scores = [t["score"] for t in main_char["traits"]]        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']    bars = ax.barh(trait_names, scores, color=colors[:len(trait_names)])    ax.set_xlabel('Score (0-1)', fontsize=12)    ax.set_title(f"{main_char['person_name']} - Big Five Profile", fontsize=14, fontweight='bold')    ax.set_xlim(0, 1)    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)        for bar, score in zip(bars, scores):        ax.text(score + 0.02, bar.get_y() + bar.get_height()/2, f'{score:.2f}',                 va='center', fontweight='bold')        plt.tight_layout()    plt.show()

---## 📈 Visualizations### Relation Type Distribution

In [ ]:
rel_counts = Counter([t["relation"] for t in triples])top_15 = rel_counts.most_common(15)fig, ax = plt.subplots(figsize=(12, 8))rels, counts = zip(*top_15)bars = ax.barh(rels, counts, color='steelblue')ax.set_xlabel('Count', fontsize=12)ax.set_title(f'Top 15 Relation Types - {novel_info["title"]}', fontsize=14, fontweight='bold')ax.invert_yaxis()for i, (r, c) in enumerate(top_15):    ax.text(c + max(counts)*0.01, i, str(c), va='center', fontweight='bold')plt.tight_layout()plt.show()print(f"Total unique relations: {len(rel_counts)}")

### Confidence Distribution

In [ ]:
confidences = [t["confidence"] for t in triples]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.hist(confidences, bins=20, color='coral', edgecolor='black', alpha=0.7)ax1.axvline(np.mean(confidences), color='red', linestyle='--',             label=f'Mean: {np.mean(confidences):.3f}', linewidth=2)ax1.set_xlabel('Confidence', fontsize=12)ax1.set_ylabel('Frequency', fontsize=12)ax1.set_title('Confidence Distribution', fontsize=14, fontweight='bold')ax1.legend()ax1.grid(alpha=0.3)ax2.boxplot(confidences, patch_artist=True,             boxprops=dict(facecolor='lightblue', alpha=0.7))ax2.set_ylabel('Confidence', fontsize=12)ax2.set_title('Confidence Statistics', fontsize=14, fontweight='bold')ax2.grid(alpha=0.3)plt.tight_layout()plt.show()print(f"Mean: {np.mean(confidences):.3f}, Median: {np.median(confidences):.3f}, Std: {np.std(confidences):.3f}")

### Big Five Trait Distribution

In [ ]:
trait_data = {    "openness": [], "conscientiousness": [], "extraversion": [],    "agreeableness": [], "neuroticism": []}for profile in traits:    for trait in profile["traits"]:        if trait["trait_name"] in trait_data:            trait_data[trait["trait_name"]].append(trait["score"])fig, ax = plt.subplots(figsize=(12, 6))labels = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']for i, (name, scores) in enumerate(trait_data.items(), 1):    if scores:        parts = ax.violinplot([scores], positions=[i], widths=0.7,                               showmeans=True, showmedians=True)        for pc in parts['bodies']:            pc.set_facecolor(colors[i-1])            pc.set_alpha(0.7)ax.set_xticks(range(1, len(labels)+1))ax.set_xticklabels(labels)ax.set_ylabel('Score (0-1)', fontsize=12)ax.set_title(f'Big Five Distribution - {novel_info["title"]}', fontsize=14, fontweight='bold')ax.set_ylim(0, 1)ax.grid(alpha=0.3, axis='y')plt.tight_layout()plt.show()

### Network: Top Character's Connections

In [ ]:
# Find most connected characterdegrees = dict(G.degree())top_node = max(degrees, key=degrees.get)if top_node in G:    ego = nx.ego_graph(G, top_node, radius=1)        fig, ax = plt.subplots(figsize=(14, 10))    pos = nx.spring_layout(ego, k=1.5, iterations=50, seed=42)        colors = []    sizes = []    for node in ego.nodes():        if node == top_node:            colors.append('#FF6B6B')            sizes.append(3000)        elif ego.nodes[node].get('entity_type') == 'person':            colors.append('#4ECDC4')            sizes.append(1500)        else:            colors.append('#95A5A6')            sizes.append(800)        nx.draw_networkx_nodes(ego, pos, node_color=colors, node_size=sizes, alpha=0.9, ax=ax)    nx.draw_networkx_labels(ego, pos, font_size=8, font_weight='bold', ax=ax)    nx.draw_networkx_edges(ego, pos, alpha=0.3, arrows=True, arrowsize=15, ax=ax)        ax.set_title(f"{top_node}'s Network ({ego.number_of_nodes()} nodes) - {novel_info['title']}",                 fontsize=14, fontweight='bold')    ax.axis('off')    plt.tight_layout()    plt.show()        print(f"Central character: {top_node} ({degrees[top_node]} connections)")

### Degree Distribution

In [ ]:
degree_vals = list(degrees.values())fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.hist(degree_vals, bins=30, color='purple', edgecolor='black', alpha=0.7)ax1.set_xlabel('Degree', fontsize=12)ax1.set_ylabel('Frequency (log scale)', fontsize=12)ax1.set_title('Degree Distribution', fontsize=14, fontweight='bold')ax1.set_yscale('log')ax1.grid(alpha=0.3)top_10 = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]nodes, degs = zip(*top_10)ax2.barh(nodes, degs, color='orange')ax2.set_xlabel('Degree', fontsize=12)ax2.set_title('Top 10 Connected Nodes', fontsize=14, fontweight='bold')ax2.invert_yaxis()for i, (n, d) in enumerate(top_10):    ax2.text(d + max(degs)*0.01, i, str(d), va='center', fontweight='bold')plt.tight_layout()plt.show()print(f"Avg degree: {np.mean(degree_vals):.2f}, Max: {max(degree_vals)} ({top_10[0][0]})")

---## 📊 Quality Metrics Summary

In [ ]:
# Comprehensive metrics tablemetrics_summary = pd.DataFrame([    {"Category": "Triples", "Metric": "Total", "Value": f"{len(triples):,}"},    {"Category": "Triples", "Metric": "Avg Confidence", "Value": f"{np.mean([t['confidence'] for t in triples]):.3f}"},    {"Category": "Triples", "Metric": "Evidence Coverage", "Value": f"{metrics['evidence_quality']['evidence_coverage']:.1%}"},    {"Category": "Relations", "Metric": "Unique Types", "Value": str(len(rel_counts))},    {"Category": "Relations", "Metric": "Shannon Entropy", "Value": f"{metrics['relation_diversity']['relation_entropy']:.3f}"},    {"Category": "Relations", "Metric": "Gini Coefficient", "Value": f"{metrics['relation_diversity']['relation_gini_coefficient']:.3f}"},    {"Category": "Personalities", "Metric": "Total Profiles", "Value": str(len(traits))},    {"Category": "Personalities", "Metric": "Avg Evidence/Trait", "Value": f"{metrics['personality_quality']['avg_evidence_per_trait']:.1f}"},    {"Category": "Graph", "Metric": "Nodes", "Value": f"{G.number_of_nodes():,}"},    {"Category": "Graph", "Metric": "Edges", "Value": f"{G.number_of_edges():,}"},    {"Category": "Graph", "Metric": "Density", "Value": f"{metrics['graph_stats']['density']:.4f}"},    {"Category": "Graph", "Metric": "Avg Path Length", "Value": f"{metrics['graph_quality']['avg_shortest_path_length']:.2f}"},])display(HTML(f"<h3>Quality Metrics - {novel_info['title']}</h3>"))display(metrics_summary)

---## 💡 Automated Insights

In [ ]:
# Generate automated insightstop_relation = rel_counts.most_common(1)[0]avg_degree = np.mean(degree_vals)graph_density = metrics['graph_stats']['density']insights = f"""### Automated Insights for {novel_info['title']}1. **Most Common Relationship**: {top_relation[0]} ({top_relation[1]} occurrences, {top_relation[1]/len(triples)*100:.1f}% of all triples)   - Suggests emphasis on {top_relation[0].lower().replace('_', ' ')} relationships2. **Network Structure**: {"Highly connected" if avg_degree > 4 else "Moderately sparse" if avg_degree > 2 else "Very sparse"}   - Average degree: {avg_degree:.2f} connections per entity   - Density: {graph_density:.4f} ({"typical for literary networks" if graph_density < 0.01 else "unusually dense"})3. **Character Depth**: {len(traits)} characters with personality profiles   - Most developed: {main_char['person_name']} ({len(main_char['traits'])} traits)   - Dominant trait: {max(main_char['traits'], key=lambda t: t['score'])['trait_name']} ({max(t['score'] for t in main_char['traits']):.2f})4. **Extraction Quality**: {metrics['evidence_quality']['evidence_coverage']:.0%} evidence coverage   - {"Excellent" if metrics['evidence_quality']['evidence_coverage'] == 1.0 else "Good"} - All triples grounded in text   - Confidence std dev: {metrics['confidence_distribution']['confidence_std_dev']:.3f} ({"well-calibrated" if metrics['confidence_distribution']['confidence_std_dev'] < 0.1 else "variable"})"""display(Markdown(insights))

---## 🔗 Resources- **Interactive Graph**: Open `{OUTPUT_DIR}/graph.html` in browser- **Raw Data**:  - Triples: `{OUTPUT_DIR}/triples_canonical.jsonl`  - Traits: `{OUTPUT_DIR}/traits_final.jsonl`  - Metrics: `{OUTPUT_DIR}/metrics.json`- **GitHub**: https://github.com/Sungchunn/Personality-Knowledge-Graph-Challenge---**Analysis completed**: {novel_info['title']} by {novel_info['author']}  **Notebook**: `demo.ipynb`  **Date**: October 20, 2025